In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

In [2]:
df = pd.read_csv('Titanic-Dataset.csv')
print(df.head())
print(df.info())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

In [3]:
print(df.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [4]:
#เติม age ด้วย median(ค่ากลาง)
df['Age'] = df['Age'].fillna(df['Age'].median())

#เติม Embarked ด้วย mode(ซ้ำเยอะสุด)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

df['Cabin'] = df['Cabin'].fillna('U').str[0]

#ดึง Title จากชื่อ เพราะบอก social status ได้ดี
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df['Title'] = df['Title'].replace(['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')
df['Title'] = df['Title'].map({'Mr':0, 'Miss':1, 'Mrs':2, 'Master':3, 'Rare':4})
df['Title'] = df['Title'].fillna(0)

#FamilySize และ IsAlone
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

#Age * Class interaction
df['Age*Class'] = df['Age'] * df['Pclass']

#ลบ column ทิ้งเพราะเป็นค่า unique ไม่ได้จำเป็นต่อ model
df.drop(['PassengerId','Name','Ticket'], axis=1, inplace=True)

In [5]:
#แปลงค่าเป็นตัวเลข
df['Sex'] = df['Sex'].map( {'male':0,'female':1} )
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
df = pd.get_dummies(df, columns=['Cabin'], drop_first=True)

In [6]:
#กำหนด target
X = df.drop('Survived', axis=1)
y = df['Survived']

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier

#Machine Learning แบบ ensemble

#random forest
rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_split=4, random_state=42)

#GradientBoosting
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)

#SVM
svm = SVC(C=1.5, kernel='rbf', probability=True, random_state=42)

In [9]:
#รวม model
ensemble = VotingClassifier(estimators=[('rf',rf), ('gb', gb), ('svm',svm)], voting='soft')

In [10]:
ensemble.fit(X_train, y_train)
pred = ensemble.predict (X_test)
accuracy = accuracy_score(y_test, pred)
print("Ensemble Accuracy =", accuracy)

Ensemble Accuracy = 0.8324022346368715


In [11]:
from tensorflow import keras
from keras import layers, regularizers

tf.random.set_seed(42)
np.random.seed(42)

model = keras.Sequential()
model.add(keras.Input(shape=(X_train.shape[1],)))

model.add(layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.0001)))
model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.3))

model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.0001)))
model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.2))

model.add(layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.0001)))
model.add(layers.Dropout(0.1))

model.add(layers.Dense(32, activation='relu'))

model.add(layers.Dense(1, activation='sigmoid'))

In [12]:
optimizer = keras.optimizers.Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

In [13]:
history = model.fit(X_train, y_train, epochs=100, batch_size=16, validation_split=0.2)
loss, accuracy = model.evaluate(X_test, y_test)
print("Neural Network Accuracy =", accuracy)

Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6415 - loss: 0.6577 - val_accuracy: 0.8392 - val_loss: 0.6312
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7821 - loss: 0.5348 - val_accuracy: 0.8322 - val_loss: 0.5848
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7944 - loss: 0.5036 - val_accuracy: 0.8392 - val_loss: 0.5468
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8207 - loss: 0.4711 - val_accuracy: 0.8392 - val_loss: 0.5052
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8348 - loss: 0.4579 - val_accuracy: 0.8531 - val_loss: 0.4687
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.8155 - loss: 0.4501 - val_accuracy: 0.8671 - val_loss: 0.4471
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.8295 - loss: 0.4315 - val_accuracy: 0.8671 - val_loss: 0.4233
Epoch 8/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8278 - loss: 0.4276 - val_accuracy: 0.